In [ ]:
import pandas as pd
import config
from rich import print
from functools import partial

In [2]:
seqno = 2025083
types = "ssq"

In [3]:
datadf = pd.read_csv(f"data/{types}/{types}_expert_{seqno}.csv",header=0)
userdf = pd.read_csv(f"data/{types}/{types}_pageno_users_{seqno}.csv",header=0)

In [4]:
userdf["userid"] = userdf["userids"].map(lambda x:eval(x))
userdf = userdf.explode("userid").drop(columns=["userids"])

In [5]:
userdf["userid"].nunique()

1500

In [6]:
datadf["userid"].nunique()

1500

In [7]:
userdf = userdf[["userid","schema","pageno"]].rename(columns={"schema":"page_schema"})
datadf = datadf[["userid","schema","numbers"]]

In [8]:
def fun(row):
    joint_numbers = set(row["numbers_joint"].split(","))
    numbers = set(row["numbers"].split(","))
    leftnumbers = list(joint_numbers - numbers)[0]
    return leftnumbers

In [9]:
schema1df = datadf.loc[lambda x:x["schema"]==config.srschema1]
schema2df = datadf.loc[lambda x:x["schema"]==config.srschema2]
schema3df = datadf.loc[lambda x:x["schema"]==config.srschema3]
jointdf = schema1df.merge(schema2df,on=["userid"],how="inner",suffixes=(None,"_joint"))
jointdf["numbers"] = jointdf.apply(fun,axis=1)
jointdf["schema"] = "R1"
datadf = pd.concat([datadf,jointdf[["userid","schema","numbers"]]],axis=0)
jointdf = schema2df.merge(schema3df,on=["userid"],how="inner",suffixes=(None,"_joint"))
jointdf["numbers"] = jointdf.apply(fun,axis=1)
jointdf["schema"] = "R11"
datadf = pd.concat([datadf,jointdf[["userid","schema","numbers"]]],axis=0)

In [10]:
df = userdf.merge(datadf,on=["userid"],how="inner")

In [11]:
cols = df.columns.values.tolist()

In [51]:
def agg_size(s):
    x = s.str.split(",").explode()
    return x.unique().size

def agg_seq(s,seq):
    x = s.str.split(",").explode()
    return set(seq) - set(x.unique())

In [53]:
blue_fun = partial(agg_seq,seq=[str(i).zfill(2) for i in range(1,17)])
red_fun = partial(agg_seq,seq=[str(i).zfill(2) for i in range(1,34)])

In [102]:
schema = config.srschema1
inddf = df.loc[lambda x:x["schema"]==schema].groupby(by=["page_schema"],as_index=False)[cols]\
    .apply(lambda x:x.reset_index(drop=True).reset_index(),include_groups=True)
inddf["pageno"] = inddf["index"] // 20

In [105]:
inddf.loc[lambda x:x["schema"]==schema].groupby(by=["page_schema","pageno"],as_index=False)\
    .agg(numbers=("numbers",agg_size),counts=("numbers","count"),seq=("numbers",red_fun))

,page_schema,pageno,numbers,counts,seq
0,凤尾两码,0,17,20,"{01, 07, 13, 24, 22, 28, 25, 17, 31, 04, 16, 1..."
1,凤尾两码,1,16,20,"{32, 07, 10, 26, 13, 12, 21, 22, 28, 25, 17, 3..."
2,凤尾两码,2,16,20,"{32, 07, 10, 26, 13, 12, 21, 22, 28, 25, 17, 3..."
3,凤尾两码,3,15,20,"{32, 07, 13, 27, 24, 12, 21, 28, 09, 18, 25, 3..."
4,凤尾两码,4,15,20,"{32, 07, 26, 27, 24, 12, 22, 28, 20, 18, 25, 3..."
...,...,...,...,...,...
970,龙头两码,70,15,20,"{32, 01, 07, 10, 27, 24, 21, 22, 28, 09, 25, 3..."
971,龙头两码,71,13,20,"{32, 26, 25, 16, 06, 07, 20, 17, 05, 08, 02, 1..."
972,龙头两码,72,15,20,"{32, 01, 26, 27, 24, 12, 21, 22, 28, 25, 04, 2..."
973,龙头两码,73,14,20,"{32, 28, 04, 06, 07, 21, 20, 17, 11, 29, 05, 3..."


In [44]:
inddf = df.loc[lambda x:x["schema"]==config.sbschemak5].groupby(by=["page_schema"],as_index=False)[cols]\
    .apply(lambda x:x.reset_index(drop=True).reset_index(),include_groups=True)
inddf["pageno"] = inddf["index"] // 20

In [87]:
inddf.loc[lambda x:x["schema"]==config.sbschemak5].groupby(by=["page_schema","pageno"],as_index=False)\
    .agg(numbers=("numbers",agg_size),counts=("numbers","count"),seq=("numbers",agg_seq))\
    .loc[lambda x:x["numbers"]==15].loc[lambda x:x["page_schema"]==config.srschema12]

,page_schema,pageno,numbers,counts,seq


In [ ]:
02 04 05 06 07 08 09 10 11 13 14 15 16

In [ ]:
01 03 12 

In [ ]:
def generate_reference(df):
    dflist = []
    for num in [9]:
        for subschema in [config.srschema1,"R1","R11"]:
            for schema in df["page_schema"].unique():
                cond = (df["page_schema"]==schema) & (df["schema"]==subschema)
                condf = df.loc[cond].reset_index(drop=True).reset_index()
                # condf["numbers"] = condf["numbers"].map(lambda x:x.split(","))
                # condf = condf.explode("numbers")
                condf["pageno"] = condf["index"] // num
                aggdf = condf.groupby(by="pageno",as_index=False).agg(numbers=("numbers",lambda x:sorted(x.unique().tolist())),ucount=("userid",pd.Series.nunique))
                aggdf["length"] = aggdf["numbers"].map(len)
                rdf = aggdf.loc[lambda x:(x["length"]==6) & (x["ucount"]==num)].copy()
                rdf["numbers"] = rdf["numbers"].map(lambda x:",".join(x))
                rdf["schema"] = schema
                rdf["num"] = num
                dflist.append(rdf)
    finaldf = pd.concat(dflist,axis=0)
    return finaldf

In [ ]:
refdf = generate_reference(df)

In [ ]:
rnumbers = [str(i).zfill(2) for i in range(1,34)]

In [ ]:
def generate_kill_numbers(refdf,rnumbers):
    def fun(numbers):
        resnums = []
        for i in numbers.split(","):
            i = int(i)
            resnums.extend([i-2,i-1,i,i+1,i+2])
        resnums = [str(i).zfill(2) for i in resnums if i>=1 and i<=33]
        return list(set(rnumbers) - set(resnums))
    refdf["knum"] = refdf["numbers"].map(fun)
    refdf = refdf.explode("knum")
    return refdf

In [ ]:
potential_knumdf = generate_kill_numbers(refdf.sample(10),rnumbers)
potential_knumdf.groupby(by=["knum"],as_index=False).agg(count=("numbers",pd.Series.nunique)).sort_values(by=["count"],ascending=[False])

In [ ]:
refdf

In [ ]:
datadf = pd.read_excel(f"data/{types}/{types}_expert_{seqno}.xlsx",header=0)

In [ ]:
dflist = []
for subschema in [config.srschemak3]:
    for schema in df["page_schema"].unique():
        cond = (df["page_schema"]==schema) & (df["schema"]==subschema)
        condf = df.loc[cond].reset_index(drop=True)
        condf["numbers"] = condf["numbers"].map(lambda x:x.split(","))
        condf = condf.explode("numbers").reset_index()
        condf["pageno"] = condf["index"] // 20
        aggdf = condf.groupby(by="pageno",as_index=False).agg(numbers=("numbers",lambda x:sorted(x.unique().tolist())))
        aggdf["length"] = aggdf["numbers"].map(len)
        rdf = aggdf.loc[lambda x:x["length"]==15].copy()
        rdf["numbers"] = rdf["numbers"].map(lambda x:",".join(x))
        rdf["schema"] = schema
        dflist.append(rdf)
finaldf = pd.concat(dflist,axis=0)

In [ ]:
rdf = df.loc[lambda x:x["schema"]==config.srschema12].copy()
rdf["numbers"] = rdf["numbers"].map(lambda x:x.split(","))
rdf = rdf.explode("numbers")
countdf = rdf.groupby(by=["page_schema","pageno"],as_index=False).agg(numbers=("numbers",pd.Series.unique),counts=("numbers",pd.Series.nunique))

In [ ]:
countdf = countdf.loc[lambda x:x["counts"]<33].copy()

In [ ]:
rnumbers = [str(i).zfill(2) for i in range(1,34)]

In [ ]:
countdf["left_numbers"] = countdf["numbers"].map(lambda x:set(rnumbers) - set(x))

In [ ]:
for s in countdf["page_schema"].unique():
    left_set = set()
    for i in countdf.loc[lambda x:x["page_schema"]==s]["left_numbers"].values.tolist():
        left_set = left_set.union(i)
    print(s,left_set)

In [ ]:
rdf = datadf[["userid","schema","latest10_hitcount"]].copy()

In [ ]:
rdf["schema"] = rdf["schema"].astype("string[pyarrow]")

In [ ]:
rdf["schema"] = rdf["schema"].map(lambda x:"ppp").astype("string[pyarrow]")

In [ ]:
cond = rdf["schema"].str.contains("五")

In [ ]:
rdf.dtypes

In [ ]:
df = pd.DataFrame()

In [ ]:
set(rnumbers) - left_set

In [ ]:
import os

In [ ]:
df1 = pd.read_csv("data/ssq_temp/2025042_00.csv")
df2 = pd.read_csv("data/ssq_temp/2025042_01.csv")
df3 = pd.read_csv("data/ssq_temp/2025042_02.csv")
df4 = pd.read_csv("data/ssq_temp/2025042_03.csv")
df5 = pd.read_csv("data/ssq_temp/2025042_100.csv")
df6 = pd.read_csv("data/ssq_temp/2025042_200.csv")
df = pd.concat([df1,df2,df3,df4,df5,df6])

In [ ]:
df.groupby(by=["numbers"],as_index=False).agg(count=("flag","count")).loc[lambda x:x["count"]==1]

In [ ]:
countdf.loc[lambda x:x["page_schema"]==config.srschemak3]

In [ ]:
finaldf

In [ ]:
03 

In [ ]:
dflist = []
for schema in df["page_schema"].unique():
    cond = (df["page_schema"]==schema) & (df["schema"]==config.srschema1)
    condf = df.loc[cond].reset_index(drop=True).reset_index()
    condf["numbers"] = condf["numbers"].map(lambda x:x.split(","))
    condf = condf.explode("numbers")
    condf["pageno"] = condf["index"] // 10
    aggdf = condf.groupby(by=["pageno","numbers"],as_index=False).agg(count=("userid","count"))
    # aggdf["length"] = aggdf["numbers"].map(len)
    rdf = aggdf.loc[lambda x:x["count"]==10]
    dflist.append(rdf)
finaldf = pd.concat(dflist,axis=0)

In [ ]:
finaldf.head(50)

In [ ]:
df.loc[cond]

In [ ]:
condf["pageno"].unique().size

In [ ]:
df

In [ ]:
for schema in df["schema"].unique():
    cond = (df["page_schema"]==schema) & (df["schema"]==config.drschema1)
    aggdf = df.loc[cond].groupby(by=["pageno","numbers"],as_index=False).agg(counts=("userid","count"))
    seqs = aggdf.loc[lambda x:x["counts"]>=5]["numbers"].unique()
    print(seqs)

In [ ]:
for schema in df["schema"].unique():
    cond = (df["page_schema"]==schema) & (df["schema"]==config.dbschema6)
    tdf = df.loc[cond].copy()
    tdf["numbers"] = tdf["numbers"].map(lambda x:x.split(","))
    tdf = tdf.explode("numbers")
    aggdf = tdf.groupby(by=["pageno","numbers"],as_index=False).agg(counts=("userid","count"))
    seqs = aggdf.loc[lambda x:x["counts"]>=17]["numbers"].unique()
    print(sorted(seqs))

In [ ]:
for schema in df["schema"].unique():
    cond = (df["page_schema"]==schema) & (df["schema"]==config.drschemak3)
    tdf = df.loc[cond].copy()
    tdf["numbers"] = tdf["numbers"].map(lambda x:x.split(","))
    tdf = tdf.explode("numbers")
    aggdf = tdf.groupby(by=["pageno","numbers"],as_index=False).agg(counts=("userid","count"))
    seqs = aggdf.loc[lambda x:x["counts"]>=6]["numbers"].unique()
    if seqs.size == 5:
        print(sorted(seqs))

In [ ]:
for schema in df["schema"].unique():
    cond = (df["page_schema"]==schema) & (df["schema"]==config.drschemak6)
    tdf = df.loc[cond].copy()
    tdf["numbers"] = tdf["numbers"].map(lambda x:x.split(","))
    tdf = tdf.explode("numbers")
    aggdf = tdf.groupby(by=["pageno","numbers"],as_index=False).agg(counts=("userid","count"))
    seqs = aggdf.loc[lambda x:x["counts"]>=9]["numbers"].unique()
    if seqs.size == 5:
        print(sorted(seqs))

In [ ]:
for schema in df["schema"].unique():
    cond = (df["page_schema"]==schema) & (df["schema"]==config.drschema10)
    tdf = df.loc[cond].copy()
    tdf["numbers"] = tdf["numbers"].map(lambda x:x.split(","))
    tdf = tdf.explode("numbers")
    aggdf = tdf.groupby(by=["pageno","numbers"],as_index=False).agg(counts=("userid","count"))
    seqs = aggdf.loc[lambda x:x["counts"]>=12]["numbers"].unique()
    if seqs.size == 5:
        print(sorted(seqs))

In [ ]:
for schema in df["schema"].unique():
    cond = (df["page_schema"]==schema) & (df["schema"]==config.drschema20)
    tdf = df.loc[cond].copy()
    tdf["numbers"] = tdf["numbers"].map(lambda x:x.split(","))
    tdf = tdf.explode("numbers")
    aggdf = tdf.groupby(by=["pageno","numbers"],as_index=False).agg(counts=("userid","count"))
    seqs = aggdf.loc[lambda x:x["counts"]>=18]["numbers"].unique()
    if seqs.size <=6:
        print(sorted(seqs))

In [ ]:
for schema in df["schema"].unique():
    cond = (df["page_schema"]==schema) & (df["schema"]==config.drschema25)
    tdf = df.loc[cond].copy()
    tdf["numbers"] = tdf["numbers"].map(lambda x:x.split(","))
    tdf = tdf.explode("numbers")
    aggdf = tdf.groupby(by=["pageno","numbers"],as_index=False).agg(counts=("userid","count"))
    seqs = aggdf.loc[lambda x:x["counts"]>=20]["numbers"].unique()
    if seqs.size == 5:
        print(sorted(seqs))

In [ ]:
for schema in df["schema"].unique():
    cond = (df["page_schema"]==schema) & (df["schema"]==config.drschema2)
    tdf = df.loc[cond].copy()
    tdf["numbers"] = tdf["numbers"].map(lambda x:x.split(","))
    tdf = tdf.explode("numbers")
    aggdf = tdf.groupby(by=["pageno","numbers"],as_index=False).agg(counts=("userid","count"))
    seqs = aggdf.loc[lambda x:x["counts"]>=7]["numbers"].unique()
    if seqs.size <= 6 and seqs.size>0:
        print(sorted(seqs))

In [ ]:
for schema in df["schema"].unique():
    cond = (df["page_schema"]==schema) & (df["schema"]==config.drschema1)
    tdf = df.loc[cond].copy()
    tdf["numbers"] = tdf["numbers"].map(lambda x:x.split(","))
    tdf = tdf.explode("numbers")
    aggdf = tdf.groupby(by=["pageno","numbers"],as_index=False).agg(counts=("userid","count"))
    seqs = aggdf.loc[lambda x:x["counts"]>=5]["numbers"].unique()
    if seqs.size <= 6 and seqs.size>0:
        print(sorted(seqs))

In [ ]:
aggdf.sort_values(by="length")

In [ ]:
df.loc[lambda x:x["page_schema"]==config.drschema1]